In [1]:
   pip install facenet-pytorch opencv-python pillow numpy

In [2]:
import os, random, shutil, copy
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from torchvision import transforms
from facenet_pytorch import MTCNN, InceptionResnetV1, extract_face, fixed_image_standardization

# Folder containing your 4 member folders (Prachi, Saachi, Hiya, Precy)
RAW = Path.home() / "Downloads" / "photos" / "photos"

# Output folder (data/ and the saved model go here). This is what you zip.
PROJECT_DIR = Path.home() / "Downloads" / "Group_x_code"

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)

DATA = Path("data")
MODEL_PATH = Path("facenet_model.pt")
EXTS = {".jpg", ".jpeg", ".png"}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def list_images(folder):
    return sorted(p for p in Path(folder).iterdir() if p.suffix.lower() in EXTS)

print("Device:", DEVICE)
print("Working in:", Path.cwd())
for m in sorted(p for p in RAW.iterdir() if p.is_dir()):
    print(m.name, "->", len(list_images(m)), "images")

C:\Users\prach\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] Disabling PyTorch because PyTorch >= 2.6 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Device: cpu
Working in: C:\Users\prach\Downloads\Group_x_code
Hiya -> 30 images
Prachi -> 30 images
Precy -> 30 images
Saachi -> 30 images


In [3]:
if DATA.exists():
    shutil.rmtree(DATA)

for member in sorted(p for p in RAW.iterdir() if p.is_dir()):
    imgs = list_images(member)
    random.shuffle(imgs)
    parts = {"train": imgs[:20], "validate": imgs[20:25], "test": imgs[25:30]}
    for name, files in parts.items():
        out = DATA / name / member.name
        out.mkdir(parents=True, exist_ok=True)
        for i, f in enumerate(files, 1):
            Image.open(f).convert("RGB").save(out / f"sample_{i}.jpg", quality=95)
    print(member.name, {k: len(v) for k, v in parts.items()})

Hiya {'train': 20, 'validate': 5, 'test': 5}
Prachi {'train': 20, 'validate': 5, 'test': 5}
Precy {'train': 20, 'validate': 5, 'test': 5}
Saachi {'train': 20, 'validate': 5, 'test': 5}


In [4]:
def random_scale_crop(im, lo=0.75):
    w, h = im.size
    s = random.uniform(lo, 1.0)
    cw, ch = int(w * s), int(h * s)
    x, y = random.randint(0, w - cw), random.randint(0, h - ch)
    return im.crop((x, y, x + cw, y + ch)).resize((w, h))

aug = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.Lambda(random_scale_crop),
    transforms.ColorJitter(brightness=0.5, contrast=0.4, saturation=0.3, hue=0.05),
    transforms.RandomApply([transforms.GaussianBlur(5, sigma=(0.1, 1.5))], p=0.3),
])

COPIES = 8
for member in sorted((DATA / "train").iterdir()):
    for old in member.glob("aug_*"):
        old.unlink()
    originals = list_images(member)
    for p in originals:
        img = Image.open(p).convert("RGB")
        for k in range(COPIES):
            aug(img).save(member / f"aug_{p.stem}_{k}.jpg", quality=95)
    print(member.name, ":", len(originals), "originals +", len(originals) * COPIES, "augmented")

Hiya : 20 originals + 160 augmented
Prachi : 20 originals + 160 augmented
Precy : 20 originals + 160 augmented
Saachi : 20 originals + 160 augmented


In [5]:
mtcnn = MTCNN(device=DEVICE, keep_all=False)
backbone = InceptionResnetV1(pretrained="vggface2").to(DEVICE)   # outputs 512-d embeddings

classes = sorted(p.name for p in (DATA / "train").iterdir() if p.is_dir())
print("Classes:", classes)

# Freeze everything, then unfreeze the last layers for fine-tuning
for p in backbone.parameters():
    p.requires_grad = False
for layer in [backbone.block8, backbone.last_linear, backbone.last_bn]:
    for p in layer.parameters():
        p.requires_grad = True

trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
total = sum(p.numel() for p in backbone.parameters())
print(f"Trainable params: {trainable:,} / {total:,}")

Classes: ['Hiya', 'Prachi', 'Precy', 'Saachi']
Trainable params: 2,519,296 / 27,910,327


In [6]:
@torch.no_grad()
def face_tensor(img):
    boxes, probs = mtcnn.detect(img)
    if boxes is None:
        return None
    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    box = boxes[int(np.argmax(areas))]          # largest face = the person at the webcam
    face = extract_face(img, box, image_size=160, margin=20).float()
    return fixed_image_standardization(face)

def build_split(split_name):
    X, y, skipped = [], [], 0
    for ci, cname in enumerate(classes):
        for p in list_images(DATA / split_name / cname):
            t = face_tensor(Image.open(p).convert("RGB"))
            if t is None:
                skipped += 1
                continue
            X.append(t); y.append(ci)
    print(f"{split_name}: {len(X)} faces used, {skipped} skipped (no face found)")
    return torch.stack(X), torch.tensor(y)

Xtr, ytr = build_split("train")
Xva, yva = build_split("validate")
Xte, yte = build_split("test")

train: 706 faces used, 14 skipped (no face found)
validate: 20 faces used, 0 skipped (no face found)
test: 20 faces used, 0 skipped (no face found)


In [7]:
def embed(X, bs=64):
    backbone.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            out.append(F.normalize(backbone(X[i:i + bs].to(DEVICE)), dim=1).cpu())
    return torch.cat(out)

def batch_hard_triplet_loss(emb, labels, margin=0.3):
    dist = torch.sqrt((2 - 2 * emb @ emb.T).clamp(min=1e-12))
    same = labels.unsqueeze(0) == labels.unsqueeze(1)
    eye = torch.eye(len(labels), dtype=torch.bool, device=emb.device)
    pos_mask = same & ~eye
    neg_mask = ~same
    hardest_pos = (dist * pos_mask.float()).max(1).values
    hardest_neg = (dist + 1e5 * (~neg_mask).float()).min(1).values
    return F.relu(hardest_pos - hardest_neg + margin).mean()

def class_centroids(X, y):
    E = embed(X)
    C = torch.stack([E[y == ci].mean(0) for ci in range(len(classes))])
    return F.normalize(C, dim=1)

def nearest_centroid(E, C):
    d = torch.cdist(E, C)
    dmin, idx = d.min(1)
    return idx, dmin

K = 8
class_idx = [torch.where(ytr == ci)[0] for ci in range(len(classes))]

def sample_batch():
    ids, labs = [], []
    for ci, idxs in enumerate(class_idx):
        pick = idxs[torch.randint(len(idxs), (K,))]
        ids.append(pick); labs += [ci] * K
    return torch.cat(ids), torch.tensor(labs)

params = [p for p in backbone.parameters() if p.requires_grad]
opt = torch.optim.Adam(params, lr=1e-4, weight_decay=1e-4)

EPOCHS, STEPS = 15, 15
best_acc, best_state = -1.0, None

for ep in range(1, EPOCHS + 1):
    backbone.eval()
    running = 0.0
    for _ in range(STEPS):
        ids, labs = sample_batch()
        emb = F.normalize(backbone(Xtr[ids].to(DEVICE)), dim=1)
        loss = batch_hard_triplet_loss(emb, labs.to(DEVICE))
        opt.zero_grad(); loss.backward(); opt.step()
        running += loss.item()

    C = class_centroids(Xtr, ytr)
    pred, _ = nearest_centroid(embed(Xva), C)
    val_acc = (pred == yva).float().mean().item()
    print(f"epoch {ep:2d} | triplet loss {running / STEPS:.4f} | val acc {val_acc:.3f}")
    if val_acc >= best_acc:
        best_acc = val_acc
        best_state = copy.deepcopy(backbone.state_dict())

backbone.load_state_dict(best_state)
print("Best val acc:", best_acc)

epoch  1 | triplet loss 0.0640 | val acc 1.000
epoch  2 | triplet loss 0.0062 | val acc 1.000
epoch  3 | triplet loss 0.0019 | val acc 1.000
epoch  4 | triplet loss 0.0123 | val acc 1.000
epoch  5 | triplet loss 0.0113 | val acc 1.000
epoch  6 | triplet loss 0.0019 | val acc 1.000
epoch  7 | triplet loss 0.0010 | val acc 1.000
epoch  8 | triplet loss 0.0000 | val acc 1.000
epoch  9 | triplet loss 0.0000 | val acc 1.000
epoch 10 | triplet loss 0.0000 | val acc 1.000
epoch 11 | triplet loss 0.0000 | val acc 1.000
epoch 12 | triplet loss 0.0002 | val acc 1.000
epoch 13 | triplet loss 0.0000 | val acc 1.000
epoch 14 | triplet loss 0.0005 | val acc 1.000
epoch 15 | triplet loss 0.0001 | val acc 1.000
Best val acc: 1.0


In [8]:
C = class_centroids(Xtr, ytr)

Ete = embed(Xte)
pred, dmin = nearest_centroid(Ete, C)
print(f"Test accuracy: {(pred == yte).float().mean().item() * 100:.1f}%")
for ci, c in enumerate(classes):
    m = yte == ci
    print(f"  {c}: {(pred[m] == ci).float().mean().item() * 100:.1f}% ({int(m.sum())} images)")

own = torch.cdist(Ete, C)[torch.arange(len(yte)), yte]
print(f"\nDistance to own centroid: min {own.min():.3f}, mean {own.mean():.3f}, max {own.max():.3f}")

all_d = torch.cdist(Ete, C)
mask = torch.ones_like(all_d, dtype=torch.bool)
mask[torch.arange(len(yte)), yte] = False
other = all_d[mask]
print(f"Distance to other members: min {other.min():.3f}, mean {other.mean():.3f}")

THRESHOLD = 0.6      # you can tune this later in the demo
torch.save({"state": backbone.state_dict(), "classes": classes,
            "centroids": C, "threshold": THRESHOLD}, MODEL_PATH)
print("\nSaved", MODEL_PATH.resolve(), "with THRESHOLD =", THRESHOLD)

Test accuracy: 100.0%
  Hiya: 100.0% (5 images)
  Prachi: 100.0% (5 images)
  Precy: 100.0% (5 images)
  Saachi: 100.0% (5 images)

Distance to own centroid: min 0.140, mean 0.257, max 0.581
Distance to other members: min 1.253, mean 1.517

Saved C:\Users\prach\Downloads\Group_x_code\facenet_model.pt with THRESHOLD = 0.6


In [11]:
import cv2

backbone.eval()
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
print("Press q in the video window to quit.")
while True:
    ok, frame = cap.read()
    if not ok:
        break
    img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    boxes, probs = mtcnn.detect(img)
    if boxes is not None:
        for box, p in zip(boxes, probs):
            if p < 0.9:
                continue
            face = fixed_image_standardization(
                extract_face(img, box, image_size=160, margin=20).float()
            ).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                e = F.normalize(backbone(face), dim=1).cpu()
            idx, d = nearest_centroid(e, C)
            dist = d.item()
            name = classes[idx.item()] if dist <= THRESHOLD else "unidentified"
            color = (0, 200, 0) if name != "unidentified" else (0, 0, 255)
            x1, y1, x2, y2 = [int(v) for v in box]
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, f"{name} (d={dist:.2f})", (x1, max(y1 - 10, 20)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    cv2.imshow("Face Recognition - FaceNet + Triplet Loss", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break
cap.release()
cv2.destroyAllWindows()

Press q in the video window to quit.


In [10]:
import os, copy
from pathlib import Path
import numpy as np
import torch, torch.nn.functional as F
from PIL import Image
from facenet_pytorch import MTCNN, InceptionResnetV1, extract_face, fixed_image_standardization

os.chdir(Path.home() / "Downloads" / "Group_x_code")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ckpt = torch.load("facenet_model.pt", map_location=DEVICE)
classes, C, THRESHOLD = ckpt["classes"], ckpt["centroids"], ckpt["threshold"]

backbone = InceptionResnetV1(pretrained=None).to(DEVICE)
backbone.load_state_dict(ckpt["state"], strict=False)   # strict=False ignores the unused 'logits' layer
backbone.eval()
mtcnn = MTCNN(device=DEVICE, keep_all=False)

def nearest_centroid(E, C):
    d = torch.cdist(E, C)
    dmin, idx = d.min(1)
    return idx, dmin

print("Loaded:", classes, "| threshold:", THRESHOLD)

Loaded: ['Hiya', 'Prachi', 'Precy', 'Saachi'] | threshold: 0.6


SyntaxError: unexpected character after line continuation character (3452497962.py, line 1)